<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;

public interface ILogger
{
    void Log(string message);
}

public class ConsoleLogger : ILogger
{
    public void Log(string message)
    {
        Console.WriteLine(message);
    }
}

public class PrefixLogger : ILogger
{
    private string _prefix;

    public PrefixLogger(string prefix)
    {
        _prefix = prefix;
    }

    public void Log(string message)
    {
        Console.WriteLine($"{_prefix} {message}");
    }
}

public class SilentLogger : ILogger
{
    public void Log(string message)
    {
    }
}

public interface IDiscountPolicy
{
    double Apply(double price, double percent);
}

public class StandardDiscountPolicy : IDiscountPolicy
{
    public double Apply(double price, double percent)
    {
        return price - price * percent / 100;
    }
}

public class LimitedDiscountPolicy : IDiscountPolicy
{
    private double _maxPercent;

    public LimitedDiscountPolicy(double maxPercent)
    {
        _maxPercent = maxPercent;
    }

    public double Apply(double price, double percent)
    {
        double real = Math.Min(percent, _maxPercent);
        return price - price * real / 100;
    }
}

public interface INotifier
{
    void Notify(string message);
}

public class ConsoleNotifier : INotifier
{
    public void Notify(string message)
    {
        Console.WriteLine($"[УВЕДОМЛЕНИЕ] {message}");
    }
}

public interface IProductServices
{
    ILogger Logger { get; }
    IDiscountPolicy DiscountPolicy { get; }
}

public class ProductServices : IProductServices
{
    public ILogger Logger { get; private set; }
    public IDiscountPolicy DiscountPolicy { get; private set; }

    public ProductServices(ILogger logger, IDiscountPolicy discountPolicy)
    {
        Logger = logger ?? throw new ArgumentNullException(nameof(logger));
        DiscountPolicy = discountPolicy ?? throw new ArgumentNullException(nameof(discountPolicy));
    }
}

public class ServiceContainer
{
    private readonly Dictionary<Type, Func<ServiceContainer, object>> _map = new Dictionary<Type, Func<ServiceContainer, object>>();

    public void RegisterSingleton<T>(T instance) where T : class
    {
        _map[typeof(T)] = c => instance;
    }

    public void Register<T>(Func<ServiceContainer, T> factory) where T : class
    {
        _map[typeof(T)] = c => factory(c);
    }

    public T Resolve<T>() where T : class
    {
        if (!_map.TryGetValue(typeof(T), out Func<ServiceContainer, object> factory))
            throw new InvalidOperationException($"Сервис {typeof(T).Name} не зарегистрирован!");
        return (T)factory(this);
    }
}

public class OrderService
{
    private ILogger _logger;
    private INotifier _notifier;
    private int _lowStockLimit;

    public int LowStockLimit
    {
        get { return _lowStockLimit; }
        set { _lowStockLimit = value; }
    }

    public OrderService(ILogger logger, INotifier notifier, int lowStockLimit = 5)
    {
        _logger = logger;
        _notifier = notifier;
        _lowStockLimit = lowStockLimit;
    }

    public bool PlaceOrder(Product product, int count)
    {
        _logger.Log($"Оформление заказа: «{product.Name}» x{count}");
        bool ok = product.Sell(count);
        if (ok && product.Quantity <= _lowStockLimit)
            _notifier.Notify($"Заканчивается товар «{product.Name}»: осталось {product.Quantity} шт.");
        return ok;
    }
}

public interface IRechargeable
{
    int BatteryLevel { get; set; }
    void Charge(int percent);
    string GetStatus();
}

public interface IWaterproof
{
    int WaterResistanceMeters { get; set; }
    bool CanSwim();
    string GetStatus();
}

public interface IPlayable
{
    int DurationMinutes { get; set; }
    void Play();
    void Pause();
}

public class Product
{
    private string _name;
    private double _price;
    private string _manufacturer;
    private string _article;
    private int _quantity;
    private string _country;
    private double _rating;
    private string _barcode;
    private IProductServices _services;

    public string Name
    {
        get { return _name; }
        set { _name = value; }
    }

    public double Price
    {
        get { return _price; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Цена не может быть отрицательной!");
            _price = value;
        }
    }

    public string Manufacturer
    {
        get { return _manufacturer; }
        set { _manufacturer = value; }
    }

    public string Article
    {
        get { return _article; }
        set { _article = value; }
    }

    public int Quantity
    {
        get { return _quantity; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Количество не может быть отрицательным!");
            _quantity = value;
        }
    }

    public string Country
    {
        get { return _country; }
        set { _country = value; }
    }

    public double Rating
    {
        get { return _rating; }
        private set { _rating = value; }
    }

    public string Barcode
    {
        get { return _barcode; }
        set { _barcode = value; }
    }

    protected ILogger Logger
    {
        get { return _services.Logger; }
    }

    protected IDiscountPolicy DiscountPolicy
    {
        get { return _services.DiscountPolicy; }
    }

    public Product(string name, double price, string manufacturer, string article, int quantity, string country,
                   IProductServices services)
    {
        _services = services ?? throw new ArgumentNullException(nameof(services));
        Name = name;
        Price = price;
        Manufacturer = manufacturer;
        Article = article;
        Quantity = quantity;
        Country = country;
        Rating = 0;
        Barcode = "000000";
    }

    public virtual string GetCategory()
    {
        return "Товар";
    }

    public virtual string GetInfo()
    {
        return $"Название: {Name}, Цена: {Price} руб., Производитель: {Manufacturer}";
    }

    public virtual void Display()
    {
        Console.WriteLine(GetInfo());
    }

    public virtual void Discount(double percent)
    {
        Price = DiscountPolicy.Apply(Price, percent);
        Logger.Log($"{Name}: скидка {percent}%. Новая цена: {Price} руб.");
    }

    public void Discount(double percent, double maxAmount)
    {
        double amount = Math.Min(Price * percent / 100, maxAmount);
        Price -= amount;
        Logger.Log($"{Name}: скидка {amount} руб. (не более {maxAmount} руб.). Новая цена: {Price} руб.");
    }

    public void AddToStock(int count)
    {
        if (count <= 0)
            throw new ArgumentOutOfRangeException(nameof(count), "Количество должно быть больше нуля!");
        Quantity += count;
        Logger.Log($"Склад пополнен: «{Name}» +{count} шт. Всего: {Quantity} шт.");
    }

    public void AddToStock(int count, string supplier)
    {
        AddToStock(count);
        Logger.Log($"Поставщик: {supplier}");
    }

    public bool Sell(int count)
    {
        if (count <= 0)
            throw new ArgumentOutOfRangeException(nameof(count), "Количество должно быть больше нуля!");
        if (count > Quantity)
        {
            Logger.Log($"Недостаточно товара «{Name}»: на складе {Quantity} шт., запрошено {count} шт.");
            return false;
        }
        Quantity -= count;
        Logger.Log($"Продано: «{Name}» {count} шт. на сумму {Price * count} руб. Остаток: {Quantity} шт.");
        return true;
    }

    public bool Sell(int count, double discountPercent)
    {
        double oldPrice = Price;
        Price = DiscountPolicy.Apply(oldPrice, discountPercent);
        bool result = Sell(count);
        Price = oldPrice;
        return result;
    }

    public double GetTotalCost()
    {
        return Price * Quantity;
    }

    public bool IsInStock()
    {
        return Quantity > 0;
    }

    public void Rate(double value)
    {
        if (value < 0 || value > 5)
            throw new ArgumentOutOfRangeException(nameof(value), "Оценка должна быть от 0 до 5!");
        Rating = Rating == 0 ? value : Math.Round((Rating + value) / 2, 1);
        Logger.Log($"«{Name}»: рейтинг {Rating}");
    }

    public double GetPriceWithVat(double vatPercent = 20)
    {
        return Price * (1 + vatPercent / 100);
    }

    public string GetStockStatus()
    {
        if (Quantity == 0) return "Нет в наличии";
        if (Quantity < 5) return "Заканчивается";
        return "В наличии";
    }

    public void ComparePriceWith(Product otherProduct)
    {
        if (Price > otherProduct.Price)
            Logger.Log($"Товар «{Name}» дороже, чем «{otherProduct.Name}» на {Price - otherProduct.Price} руб.");
        else if (Price < otherProduct.Price)
            Logger.Log($"Товар «{Name}» дешевле, чем «{otherProduct.Name}» на {otherProduct.Price - Price} руб.");
        else
            Logger.Log($"Товары «{Name}» и «{otherProduct.Name}» стоят одинаково.");
    }

    public void ComparePriceWith(double price)
    {
        if (Price > price)
            Logger.Log($"Товар «{Name}» дороже {price} руб. на {Price - price} руб.");
        else if (Price < price)
            Logger.Log($"Товар «{Name}» дешевле {price} руб. на {price - Price} руб.");
        else
            Logger.Log($"Товар «{Name}» стоит ровно {price} руб.");
    }

    public void ApplyBulkDiscount(Product secondProduct, double bundleDiscountPercent)
    {
        Logger.Log($"[Комплект «{Name}» + «{secondProduct.Name}»]");
        Discount(bundleDiscountPercent);
        secondProduct.Discount(bundleDiscountPercent);
    }
}

public class Electronics : Product
{
    private int _warrantyPeriod;
    private string _model;
    private int _powerWatts;
    private string _color;
    private string _energyClass;
    private int _voltage;

    public int WarrantyPeriod
    {
        get { return _warrantyPeriod; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Гарантия не может быть отрицательной!");
            _warrantyPeriod = value;
        }
    }

    public string Model
    {
        get { return _model; }
        set { _model = value; }
    }

    public int PowerWatts
    {
        get { return _powerWatts; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Мощность не может быть отрицательной!");
            _powerWatts = value;
        }
    }

    public string Color
    {
        get { return _color; }
        set { _color = value; }
    }

    public string EnergyClass
    {
        get { return _energyClass; }
        set { _energyClass = value; }
    }

    public int Voltage
    {
        get { return _voltage; }
        set { _voltage = value; }
    }

    public Electronics(string name, double price, string manufacturer, string article, int quantity, string country,
                       int warrantyPeriod, string model, int powerWatts, string color, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, services)
    {
        WarrantyPeriod = warrantyPeriod;
        Model = model;
        PowerWatts = powerWatts;
        Color = color;
        EnergyClass = "A";
        Voltage = 220;
    }

    public override string GetCategory()
    {
        return "Электроника";
    }

    public override void Display()
    {
        Console.WriteLine(GetInfo());
        Console.WriteLine(GetTechInfo());
    }

    public override void Discount(double percent)
    {
        double extra = WarrantyPeriod > 12 ? 5 : 0;
        base.Discount(percent + extra);
    }

    public bool IsUnderWarranty(int monthsUsed)
    {
        return monthsUsed <= WarrantyPeriod;
    }

    public double CalculateEnergyCost(double hours, double tariffPerKWh)
    {
        return PowerWatts / 1000.0 * hours * tariffPerKWh;
    }

    public void ExtendWarranty(int months)
    {
        WarrantyPeriod += months;
        Logger.Log($"Гарантия «{Name}» продлена на {months} мес. Теперь: {WarrantyPeriod} мес.");
    }

    public bool IsEnergyEfficient()
    {
        return EnergyClass == "A" || EnergyClass == "A+" || EnergyClass == "A++";
    }

    public virtual string GetTechInfo()
    {
        return $"Модель: {Model}, Мощность: {PowerWatts} Вт, Цвет: {Color}, Гарантия: {WarrantyPeriod} мес.";
    }
}

public class Clothing : Product
{
    private string _size;
    private string _material;
    private string _season;
    private string _color;
    private string _gender;
    private bool _isOnSale;

    public string Size
    {
        get { return _size; }
        set { _size = value; }
    }

    public string Material
    {
        get { return _material; }
        set { _material = value; }
    }

    public string Season
    {
        get { return _season; }
        set { _season = value; }
    }

    public string Color
    {
        get { return _color; }
        set { _color = value; }
    }

    public string Gender
    {
        get { return _gender; }
        set { _gender = value; }
    }

    public bool IsOnSale
    {
        get { return _isOnSale; }
        private set { _isOnSale = value; }
    }

    public Clothing(string name, double price, string manufacturer, string article, int quantity, string country,
                    string size, string material, string season, string color, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, services)
    {
        Size = size;
        Material = material;
        Season = season;
        Color = color;
        Gender = "Унисекс";
        IsOnSale = false;
    }

    public override string GetCategory()
    {
        return "Одежда";
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Размер: {Size}, Материал: {Material}";
    }

    public bool IsSuitableForSeason(string season)
    {
        return Season.Equals(season, StringComparison.OrdinalIgnoreCase) || Season.Equals("Всесезонная", StringComparison.OrdinalIgnoreCase);
    }

    public string GetCareInstructions()
    {
        switch (Material.ToLower())
        {
            case "хлопок": return "Стирка при 40°C, можно гладить.";
            case "шерсть": return "Только ручная стирка при 30°C.";
            case "синтетика": return "Деликатная стирка, не гладить горячим утюгом.";
            default: return "Следуйте информации на ярлыке.";
        }
    }

    public void ChangeSize(string newSize)
    {
        Logger.Log($"Размер «{Name}» изменён: {Size} -> {newSize}");
        Size = newSize;
    }

    public void PutOnSale(double percent)
    {
        Discount(percent);
        IsOnSale = true;
    }

    public string GetSizeInfo()
    {
        return $"{Name}: размер {Size}, предназначено для: {Gender}";
    }
}

public class Books : Product
{
    private string _author;
    private string _genre;
    private int _pages;
    private int _year;
    private string _publisher;
    private string _isbn;

    public string Author
    {
        get { return _author; }
        set { _author = value; }
    }

    public string Genre
    {
        get { return _genre; }
        set { _genre = value; }
    }

    public int Pages
    {
        get { return _pages; }
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Количество страниц должно быть положительным!");
            _pages = value;
        }
    }

    public int Year
    {
        get { return _year; }
        set { _year = value; }
    }

    public string Publisher
    {
        get { return _publisher; }
        set { _publisher = value; }
    }

    public string Isbn
    {
        get { return _isbn; }
        set { _isbn = value; }
    }

    public Books(string name, double price, string manufacturer, string article, int quantity, string country,
                 string author, string genre, int pages, int year, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, services)
    {
        Author = author;
        Genre = genre;
        Pages = pages;
        Year = year;
        Publisher = manufacturer;
        Isbn = "не указан";
    }

    public override string GetCategory()
    {
        return "Книги";
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Автор: {Author}, Жанр: {Genre}";
    }

    public double GetReadingTime(int pagesPerHour = 30)
    {
        return (double)Pages / pagesPerHour;
    }

    public bool IsClassic()
    {
        return DateTime.Now.Year - Year > 100;
    }

    public string GetBookCard()
    {
        return $"«{Name}» — {Author}, жанр: {Genre}, {Pages} стр., {Year} г.";
    }

    public bool IsLongBook()
    {
        return Pages > 500;
    }

    public string GetCitation()
    {
        return $"{Author}. {Name}. — {Publisher}, {Year}. — {Pages} с. ISBN: {Isbn}";
    }
}

public class Smartphone : Electronics
{
    private int _storageGb;
    private double _screenInches;
    private string _operatingSystem;
    private int _batteryMah;
    private bool _hasNfc;

    public int StorageGb
    {
        get { return _storageGb; }
        set { _storageGb = value; }
    }

    public double ScreenInches
    {
        get { return _screenInches; }
        set { _screenInches = value; }
    }

    public string OperatingSystem
    {
        get { return _operatingSystem; }
        set { _operatingSystem = value; }
    }

    public int BatteryMah
    {
        get { return _batteryMah; }
        set { _batteryMah = value; }
    }

    public bool HasNfc
    {
        get { return _hasNfc; }
        set { _hasNfc = value; }
    }

    public Smartphone(string name, double price, string manufacturer, string article, int quantity, string country,
                      int warrantyPeriod, string model, int powerWatts, string color,
                      int storageGb, double screenInches, string operatingSystem, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, warrantyPeriod, model, powerWatts, color, services)
    {
        StorageGb = storageGb;
        ScreenInches = screenInches;
        OperatingSystem = operatingSystem;
        BatteryMah = 4500;
        HasNfc = true;
    }

    public override string GetCategory()
    {
        return "Электроника / Смартфоны";
    }

    public override string GetTechInfo()
    {
        return base.GetTechInfo() + $", Память: {StorageGb} ГБ, Экран: {ScreenInches}\", ОС: {OperatingSystem}";
    }

    public void MakeCall(string number)
    {
        Logger.Log($"{Name} ({Model}): звонок на номер {number}...");
    }

    public void MakeCall(string number, int seconds)
    {
        MakeCall(number);
        Logger.Log($"Длительность звонка: {seconds} сек.");
    }

    public void UpgradeStorage(int extraGb)
    {
        StorageGb += extraGb;
        Logger.Log($"Память {Name} увеличена на {extraGb} ГБ. Теперь: {StorageGb} ГБ.");
    }

    public void SyncWith(SmartWatch watch)
    {
        IRechargeable rechargeable = watch;
        Logger.Log($"{Name} синхронизирован с часами «{watch.Name}» (заряд часов: {rechargeable.BatteryLevel}%).");
        if (rechargeable.BatteryLevel < 20)
            Logger.Log("Внимание: у часов низкий заряд!");
    }

    public double EstimateTalkTimeHours()
    {
        return BatteryMah / 300.0;
    }

    public void SetNfc(bool enabled)
    {
        HasNfc = enabled;
        Logger.Log($"{Name}: NFC {(enabled ? "включён" : "выключен")}.");
    }
}

public class GamingSmartphone : Smartphone
{
    private int _refreshRate;
    private bool _hasCooling;
    private string _gpu;
    private int _boostLevel;

    public int RefreshRate
    {
        get { return _refreshRate; }
        set { _refreshRate = value; }
    }

    public bool HasCooling
    {
        get { return _hasCooling; }
        set { _hasCooling = value; }
    }

    public string Gpu
    {
        get { return _gpu; }
        set { _gpu = value; }
    }

    public int BoostLevel
    {
        get { return _boostLevel; }
        private set { _boostLevel = value; }
    }

    public GamingSmartphone(string name, double price, string manufacturer, string article, int quantity, string country,
                            int warrantyPeriod, string model, int powerWatts, string color,
                            int storageGb, double screenInches, string operatingSystem,
                            int refreshRate, bool hasCooling, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, warrantyPeriod, model, powerWatts, color,
               storageGb, screenInches, operatingSystem, services)
    {
        RefreshRate = refreshRate;
        HasCooling = hasCooling;
        Gpu = "Adreno";
        BoostLevel = 0;
    }

    public override string GetCategory()
    {
        return "Электроника / Игровые смартфоны";
    }

    public override string GetTechInfo()
    {
        return base.GetTechInfo() + $", Частота экрана: {RefreshRate} Гц, Охлаждение: {(HasCooling ? "есть" : "нет")}, GPU: {Gpu}";
    }

    public void LaunchGame(string game)
    {
        Logger.Log($"{Name}: запуск игры «{game}» на {RefreshRate} Гц" + (HasCooling ? " (охлаждение включено)." : "."));
    }

    public void Boost(int level)
    {
        if (!HasCooling && level > 1)
        {
            Logger.Log($"{Name}: без охлаждения максимальный уровень буста — 1.");
            level = 1;
        }
        BoostLevel = level;
        Logger.Log($"{Name}: уровень буста {BoostLevel}.");
    }

    public bool IsHighRefreshRate()
    {
        return RefreshRate >= 120;
    }
}

public class SmartWatch : Electronics, IRechargeable, IWaterproof
{
    private int _batteryLevel;
    private int _waterResistanceMeters;
    private string _strapMaterial;
    private int _stepCount;
    private bool _hasGps;

    public string StrapMaterial
    {
        get { return _strapMaterial; }
        set { _strapMaterial = value; }
    }

    public int StepCount
    {
        get { return _stepCount; }
        private set { _stepCount = value; }
    }

    public bool HasGps
    {
        get { return _hasGps; }
        set { _hasGps = value; }
    }

    public SmartWatch(string name, double price, string manufacturer, string article, int quantity, string country,
                      int warrantyPeriod, string model, int powerWatts, string color,
                      int batteryLevel, int waterResistanceMeters, string strapMaterial, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, warrantyPeriod, model, powerWatts, color, services)
    {
        SetBattery(batteryLevel);
        _waterResistanceMeters = waterResistanceMeters;
        StrapMaterial = strapMaterial;
        StepCount = 0;
        HasGps = true;
    }

    private void SetBattery(int value)
    {
        if (value < 0 || value > 100)
            throw new ArgumentOutOfRangeException(nameof(value), "Заряд должен быть от 0 до 100%!");
        _batteryLevel = value;
    }

    private bool CheckSwim()
    {
        return _waterResistanceMeters >= 50;
    }

    int IRechargeable.BatteryLevel
    {
        get { return _batteryLevel; }
        set { SetBattery(value); }
    }

    void IRechargeable.Charge(int percent)
    {
        SetBattery(Math.Min(100, _batteryLevel + percent));
        Logger.Log($"{Name}: зарядка +{percent}%. Текущий заряд: {_batteryLevel}%.");
    }

    string IRechargeable.GetStatus()
    {
        return $"Батарея: {_batteryLevel}%";
    }

    int IWaterproof.WaterResistanceMeters
    {
        get { return _waterResistanceMeters; }
        set { _waterResistanceMeters = value; }
    }

    bool IWaterproof.CanSwim()
    {
        return CheckSwim();
    }

    string IWaterproof.GetStatus()
    {
        return $"Водозащита: {_waterResistanceMeters} м ({(CheckSwim() ? "можно плавать" : "только брызги")})";
    }

    public override string GetCategory()
    {
        return "Электроника / Умные часы";
    }

    public override string GetTechInfo()
    {
        return base.GetTechInfo() + $", Заряд: {_batteryLevel}%, Водозащита: {_waterResistanceMeters} м, Ремешок: {StrapMaterial}";
    }

    public void AddSteps(int steps)
    {
        StepCount += steps;
        Logger.Log($"{Name}: шагов за день — {StepCount}.");
    }

    public double GetGoalProgress(int goal)
    {
        return Math.Min(100.0, StepCount * 100.0 / goal);
    }
}

public class AudioBook : Books, IPlayable
{
    private int _durationMinutes;
    private string _narrator;
    private double _playbackSpeed;
    private int _currentMinute;

    public string Narrator
    {
        get { return _narrator; }
        set { _narrator = value; }
    }

    public double PlaybackSpeed
    {
        get { return _playbackSpeed; }
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException(nameof(value), "Скорость должна быть положительной!");
            _playbackSpeed = value;
        }
    }

    public int CurrentMinute
    {
        get { return _currentMinute; }
        private set { _currentMinute = value; }
    }

    public AudioBook(string name, double price, string manufacturer, string article, int quantity, string country,
                     string author, string genre, int pages, int year,
                     int durationMinutes, string narrator, IProductServices services)
        : base(name, price, manufacturer, article, quantity, country, author, genre, pages, year, services)
    {
        _durationMinutes = durationMinutes;
        Narrator = narrator;
        PlaybackSpeed = 1.0;
        CurrentMinute = 0;
    }

    int IPlayable.DurationMinutes
    {
        get { return _durationMinutes; }
        set { _durationMinutes = value; }
    }

    void IPlayable.Play()
    {
        Logger.Log($"Воспроизведение «{Name}» (читает {Narrator}), длительность {_durationMinutes / 60} ч {_durationMinutes % 60} мин, скорость x{PlaybackSpeed}.");
    }

    void IPlayable.Pause()
    {
        Logger.Log($"«{Name}»: пауза на {CurrentMinute}-й минуте.");
    }

    public override string GetCategory()
    {
        return "Книги / Аудиокниги";
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Чтец: {Narrator}";
    }

    public void Rewind(int minutes)
    {
        CurrentMinute = Math.Max(0, Math.Min(_durationMinutes, CurrentMinute + minutes));
        Logger.Log($"«{Name}»: перемотка, текущая минута — {CurrentMinute}.");
    }

    public double GetProgress()
    {
        return CurrentMinute * 100.0 / _durationMinutes;
    }
}

public class Inventory<T> where T : Product
{
    private string _title;
    private List<T> _items = new List<T>();
    private ILogger _logger;

    public string Title
    {
        get { return _title; }
        set { _title = value; }
    }

    public int Count
    {
        get { return _items.Count; }
    }

    public Inventory(string title, ILogger logger)
    {
        Title = title;
        _logger = logger ?? throw new ArgumentNullException(nameof(logger));
    }

    public void Add(T item)
    {
        _items.Add(item);
        _logger.Log($"[{Title}] добавлено: {item.Name}");
    }

    public bool Remove(T item)
    {
        bool removed = _items.Remove(item);
        if (removed)
            _logger.Log($"[{Title}] удалено: {item.Name}");
        return removed;
    }

    public T FindByName(string name)
    {
        return _items.Find(p => p.Name == name);
    }

    public T GetMostExpensive()
    {
        T result = null;
        foreach (T item in _items)
        {
            if (result == null || item.Price > result.Price)
                result = item;
        }
        return result;
    }

    public double GetTotalCost()
    {
        double sum = 0;
        foreach (T item in _items)
            sum += item.GetTotalCost();
        return sum;
    }

    public List<T> Filter(Func<T, bool> predicate)
    {
        return _items.FindAll(p => predicate(p));
    }

    public void ApplyDiscountToAll(double percent)
    {
        foreach (T item in _items)
            item.Discount(percent);
    }

    public void ShowAll()
    {
        Console.WriteLine($"=== {Title} (позиций: {Count}) ===");
        foreach (T item in _items)
        {
            Console.WriteLine($"[{item.GetCategory()}]");
            item.Display();
        }
    }
}

public class Bundle<TFirst, TSecond> where TFirst : Product where TSecond : Product
{
    private double _discountPercent;

    public TFirst First { get; set; }
    public TSecond Second { get; set; }

    public double DiscountPercent
    {
        get { return _discountPercent; }
        set { _discountPercent = value; }
    }

    public Bundle(TFirst first, TSecond second, double discountPercent)
    {
        First = first;
        Second = second;
        DiscountPercent = discountPercent;
    }

    public double GetPrice()
    {
        return (First.Price + Second.Price) * (1 - DiscountPercent / 100);
    }

    public void Show()
    {
        Console.WriteLine($"Комплект: «{First.Name}» + «{Second.Name}», скидка {DiscountPercent}%, цена: {GetPrice()} руб.");
    }
}

public static class ProductUtils
{
    public static T GetCheaper<T>(T first, T second) where T : Product
    {
        return first.Price <= second.Price ? first : second;
    }
}

ServiceContainer container = new ServiceContainer();
container.RegisterSingleton<ILogger>(new PrefixLogger("[LOG]"));
container.RegisterSingleton<IDiscountPolicy>(new LimitedDiscountPolicy(30));
container.RegisterSingleton<INotifier>(new ConsoleNotifier());
container.Register<IProductServices>(c => new ProductServices(c.Resolve<ILogger>(), c.Resolve<IDiscountPolicy>()));
container.Register<OrderService>(c => new OrderService(c.Resolve<ILogger>(), c.Resolve<INotifier>()));

IProductServices services = container.Resolve<IProductServices>();

Electronics tv = new Electronics("Телевизор", 40000, "LG", "EL-001", 5, "Корея", 24, "OLED55", 120, "Чёрный", services);
Clothing tshirt = new Clothing("Футболка", 1500, "Nike", "CL-010", 20, "Вьетнам", "L", "хлопок", "Лето", "Белый", services);
Books book = new Books("Война и мир", 800, "Эксмо", "BK-100", 15, "Россия", "Л.Н. Толстой", "Роман", 1225, 1869, services);
Smartphone phone = new Smartphone("Смартфон", 50000, "Samsung", "EL-200", 8, "Корея", 24, "Galaxy S", 15, "Серый", 256, 6.5, "Android", services);
GamingSmartphone gamePhone = new GamingSmartphone("Игровой смартфон", 80000, "ASUS", "EL-300", 3, "Тайвань", 12, "ROG Phone", 30, "Чёрный", 512, 6.8, "Android", 165, true, services);
SmartWatch watch = new SmartWatch("Умные часы", 15000, "Apple", "EL-400", 12, "Китай", 12, "Watch 9", 2, "Серебристый", 15, 50, "Силикон", services);

IProductServices quietServices = new ProductServices(new SilentLogger(), new StandardDiscountPolicy());
AudioBook audio = new AudioBook("Мастер и Маргарита (аудио)", 600, "Строки", "BK-200", 100, "Россия", "М.А. Булгаков", "Роман", 480, 1967, 1230, "Сергей Чонишвили", quietServices);

Console.WriteLine("===== Полиморфизм: перекрытие методов =====");
List<Product> products = new List<Product> { tv, tshirt, book, phone, gamePhone, watch, audio };
foreach (Product p in products)
{
    Console.WriteLine($"[{p.GetCategory()}]");
    p.Display();
    Console.WriteLine();
}

Console.WriteLine("===== Полиморфизм: перегрузка методов =====");
tv.Sell(2);
tv.Sell(1, 10);
tv.AddToStock(5);
tv.AddToStock(10, "ООО Техноснаб");
tv.Discount(10);
tshirt.Discount(20, 200);
tv.ComparePriceWith(tshirt);
tv.ComparePriceWith(30000);
phone.MakeCall("+7 900 123-45-67");
phone.MakeCall("+7 900 123-45-67", 45);

Console.WriteLine("\n===== Явная реализация интерфейсов =====");
IRechargeable rechargeable = watch;
IWaterproof waterproof = watch;
rechargeable.Charge(50);
Console.WriteLine(rechargeable.GetStatus());
Console.WriteLine(waterproof.GetStatus());
Console.WriteLine($"Можно плавать: {waterproof.CanSwim()}");

IPlayable player = audio;
player.Play();
player.Pause();
audio.Rewind(30);
Console.WriteLine($"Прогресс аудиокниги (тихий логгер, но Console работает): {audio.GetProgress():F1}%");

gamePhone.LaunchGame("Genshin Impact");
gamePhone.Boost(3);
phone.SyncWith(watch);

Console.WriteLine("\n===== Управление зависимостями =====");
OrderService orders = container.Resolve<OrderService>();
orders.PlaceOrder(gamePhone, 2);
orders.PlaceOrder(tv, 1);

tv.Rate(4.5);
Console.WriteLine($"Цена телевизора с НДС: {tv.GetPriceWithVat()} руб.");
tshirt.PutOnSale(50);
watch.AddSteps(7500);
Console.WriteLine($"Прогресс цели по шагам: {watch.GetGoalProgress(10000)}%");

Console.WriteLine("\n===== Generic классы =====");
ILogger logger = container.Resolve<ILogger>();
Inventory<Product> store = new Inventory<Product>("Главный склад", logger);
foreach (Product p in products)
    store.Add(p);

Inventory<Electronics> techStore = new Inventory<Electronics>("Склад электроники", new SilentLogger());
techStore.Add(tv);
techStore.Add(phone);
techStore.Add(gamePhone);
techStore.Add(watch);

Console.WriteLine($"Самый дорогой в электронике: {techStore.GetMostExpensive().Name}");
Console.WriteLine($"Стоимость склада электроники: {techStore.GetTotalCost()} руб.");
techStore.ApplyDiscountToAll(10);
Console.WriteLine($"Стоимость после скидки: {techStore.GetTotalCost()} руб.");

foreach (Product cheap in store.Filter(p => p.Price < 2000))
    Console.WriteLine($"Дешёвый товар: {cheap.Name} — {cheap.Price} руб.");

Product found = store.FindByName("Футболка");
Console.WriteLine($"Поиск: {found.GetInfo()}");

Bundle<Smartphone, SmartWatch> bundle = new Bundle<Smartphone, SmartWatch>(phone, watch, 10);
bundle.Show();

Product cheaper = ProductUtils.GetCheaper<Product>(tv, tshirt);
Console.WriteLine($"Дешевле: {cheaper.Name}");
Electronics cheaperTech = ProductUtils.GetCheaper<Electronics>(phone, watch);
Console.WriteLine($"Дешевле из электроники: {cheaperTech.Name}");


===== Полиморфизм: перекрытие методов =====
[Электроника]
Название: Телевизор, Цена: 40000 руб., Производитель: LG
Модель: OLED55, Мощность: 120 Вт, Цвет: Чёрный, Гарантия: 24 мес.

[Одежда]
Название: Футболка, Цена: 1500 руб., Производитель: Nike, Размер: L, Материал: хлопок

[Книги]
Название: Война и мир, Цена: 800 руб., Производитель: Эксмо, Автор: Л.Н. Толстой, Жанр: Роман

[Электроника / Смартфоны]
Название: Смартфон, Цена: 50000 руб., Производитель: Samsung
Модель: Galaxy S, Мощность: 15 Вт, Цвет: Серый, Гарантия: 24 мес., Память: 256 ГБ, Экран: 6,5", ОС: Android

[Электроника / Игровые смартфоны]
Название: Игровой смартфон, Цена: 80000 руб., Производитель: ASUS
Модель: ROG Phone, Мощность: 30 Вт, Цвет: Чёрный, Гарантия: 12 мес., Память: 512 ГБ, Экран: 6,8", ОС: Android, Частота экрана: 165 Гц, Охлаждение: есть, GPU: Adreno

[Электроника / Умные часы]
Название: Умные часы, Цена: 15000 руб., Производитель: Apple
Модель: Watch 9, Мощность: 2 Вт, Цвет: Серебристый, Гарантия: 12 мес.